# Phase 2 Substrate-Comparison Grid — Colab T4

Runs the **5-cell Phase-2 grid** for one state at a time (FL → CA → TX): substrate linear probe (Leg I) + cat STL × 2 substrates (Leg II.1) + reg STL × 2 substrates (Leg II.2) + MTL B3 counterfactual on HGI (Leg III).

**See:** `docs/studies/check2hgi/PHASE2_TRACKER.md` (work queue + acceptance) · `docs/studies/check2hgi/research/SUBSTRATE_COMPARISON_PLAN.md` (3-leg framework) · `docs/COLAB_GUIDE.md` (operational guide).

Grid (per state): F36a probe → F36b cat-STL × 2 → F36c reg-STL × 2 → F36d MTL counterfactual. Wall-clock on T4: ~50 min × 5 ≈ ~3 h for FL.

Sequential — each cell waits for prior subprocess to exit, then launches next. All logs land on Drive.

---
## ① Mount Drive + paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

DRIVE_ROOT   = Path('/content/drive/MyDrive/mestrado/PoiMtlNet')
DATA_ROOT    = DRIVE_ROOT / 'data'
OUTPUT_DIR   = DRIVE_ROOT / 'output'
RESULTS_ROOT = DRIVE_ROOT / 'results'
LOGS_DIR     = RESULTS_ROOT / 'phase2_logs'
LOGS_DIR.mkdir(parents=True, exist_ok=True)

os.environ['DATA_ROOT']   = str(DRIVE_ROOT)
os.environ['OUTPUT_DIR']  = str(OUTPUT_DIR)
os.environ['PYTHONPATH']  = 'src'

for p in (DATA_ROOT, OUTPUT_DIR, RESULTS_ROOT):
    print(f"  {'OK' if p.exists() else 'MISSING'}  {p}")

---
## ② Clone repo + checkout `worktree-check2hgi-mtl`

This branch contains both the Phase-2 substrate-comparison scripts (`p1_region_head_ablation.py`, `probe/substrate_linear_probe.py`, `probe/build_hgi_next_region.py`) and the `feat/colab-gpu-perf` patches (already merged in via rebase).

In [ ]:
REPO_DIR = Path('/content/PoiMtlNet')
BRANCH = 'worktree-check2hgi-mtl'

if not REPO_DIR.exists():
    !git clone https://github.com/VitorHugoOli/PoiMtlNet.git {REPO_DIR}

%cd {REPO_DIR}
!git fetch --quiet origin {BRANCH}
!git checkout {BRANCH}
!git pull --ff-only
!git log --oneline -3

import subprocess
for path in ('scripts/p1_region_head_ablation.py',
             'scripts/probe/substrate_linear_probe.py',
             'scripts/probe/build_hgi_next_region.py'):
    print(f"  {'OK' if Path(path).exists() else 'MISSING'}  {path}")
for flag in ('task-a-input-type', 'task-set'):
    n = int(subprocess.run(['grep', '-c', flag, 'scripts/train.py'],
                            capture_output=True, text=True).stdout.strip() or 0)
    print(f"  {'OK' if n > 0 else 'MISSING'} --{flag}")

---
## ③ Install deps + confirm CUDA

In [ ]:
!pip install -q -r requirements_colab.txt

import torch
_tv = torch.__version__
_pyg_url = f'https://data.pyg.org/whl/torch-{_tv}.html'
!pip uninstall -q -y torch-scatter torch-sparse torch-cluster torch-spline-conv 2>/dev/null
!pip install -q torch-scatter torch-sparse torch-cluster torch-spline-conv -f {_pyg_url}

print(f'\nDevice: {"cuda" if torch.cuda.is_available() else "cpu"}  ·  torch {_tv}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}  ·  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

---
## ④ Configure the run

Edit `STATE` for your target. Defines the 5-cell grid with all CLI invocations the launcher in §⑤ will execute sequentially.

In [ ]:
# ── EDIT HERE ────────────────────────────────────────────────────────────────
STATE  = 'florida'      # 'florida' | 'california' | 'texas'
FOLDS  = 5
EPOCHS = 50
SEED   = 42
# ─────────────────────────────────────────────────────────────────────────────

UPSTATE = STATE.upper()
TRANSITION_PATH = OUTPUT_DIR / 'check2hgi' / STATE / 'region_transition_log.pt'

# Pre-flight: required artefacts per PHASE2_TRACKER.md §3.
checks = {
    'C2HGI embeddings':         OUTPUT_DIR / 'check2hgi' / STATE / 'embeddings.parquet',
    'C2HGI region_embeddings':  OUTPUT_DIR / 'check2hgi' / STATE / 'region_embeddings.parquet',
    'C2HGI transition_log':     TRANSITION_PATH,
    'C2HGI input/next':         OUTPUT_DIR / 'check2hgi' / STATE / 'input' / 'next.parquet',
    'C2HGI input/next_region':  OUTPUT_DIR / 'check2hgi' / STATE / 'input' / 'next_region.parquet',
    'HGI embeddings':           OUTPUT_DIR / 'hgi' / STATE / 'embeddings.parquet',
    'HGI input/next':           OUTPUT_DIR / 'hgi' / STATE / 'input' / 'next.parquet',
    'HGI input/next_region':    OUTPUT_DIR / 'hgi' / STATE / 'input' / 'next_region.parquet',
}
missing = [n for n, p in checks.items() if not p.exists()]
for name, p in checks.items():
    print(f"  {'OK' if p.exists() else 'MISSING'}  {name}")

# HGI input/next_region.parquet is built deterministically from C2HGI labels — auto-build if missing.
hgi_nr = OUTPUT_DIR / 'hgi' / STATE / 'input' / 'next_region.parquet'
if not hgi_nr.exists() and (OUTPUT_DIR / 'hgi' / STATE / 'input' / 'next.parquet').exists():
    print(f'\nBuilding HGI input/next_region.parquet for {STATE} (deterministic from C2HGI labels)...')
    !python3 scripts/probe/build_hgi_next_region.py --state {STATE}
    missing = [n for n, p in checks.items() if not p.exists()]

if missing:
    raise RuntimeError(f'Missing artefacts (block Phase-2 launch): {missing}')
print('\nAll required artefacts present — ready to launch.')

# ── 5-cell experiment grid ───────────────────────────────────────────────────
EXPERIMENTS = [
    # F36a — substrate linear probe (Leg I, head-free, CPU-only, fast)
    {'id': 'F36a_probe_check2hgi',
     'cmd': f'python3 scripts/probe/substrate_linear_probe.py --state {STATE} --engine check2hgi'},
    {'id': 'F36a_probe_hgi',
     'cmd': f'python3 scripts/probe/substrate_linear_probe.py --state {STATE} --engine hgi'},

    # F36b — cat STL matched-head (next_gru) × {check2hgi, hgi}
    {'id': f'F36b_cat_stl_check2hgi_{UPSTATE}',
     'cmd': (f'python3 -u scripts/train.py --task next --state {STATE} --engine check2hgi '
             f'--model next_gru --folds {FOLDS} --epochs {EPOCHS} --seed {SEED} --no-checkpoints')},
    {'id': f'F36b_cat_stl_hgi_{UPSTATE}',
     'cmd': (f'python3 -u scripts/train.py --task next --state {STATE} --engine hgi '
             f'--model next_gru --folds {FOLDS} --epochs {EPOCHS} --seed {SEED} --no-checkpoints')},

    # F36c — reg STL matched-head (next_getnext_hard) × {check2hgi, hgi}
    {'id': f'F36c_reg_stl_check2hgi_{UPSTATE}',
     'cmd': (f'python3 -u scripts/p1_region_head_ablation.py --state {STATE} '
             f'--heads next_getnext_hard --folds {FOLDS} --epochs {EPOCHS} --seed {SEED} '
             f'--input-type region --region-emb-source check2hgi '
             f'--override-hparams d_model=256 num_heads=8 '
             f'transition_path={TRANSITION_PATH} '
             f'--tag STL_{UPSTATE}_check2hgi_reg_gethard_{FOLDS}f{EPOCHS}ep')},
    {'id': f'F36c_reg_stl_hgi_{UPSTATE}',
     'cmd': (f'python3 -u scripts/p1_region_head_ablation.py --state {STATE} '
             f'--heads next_getnext_hard --folds {FOLDS} --epochs {EPOCHS} --seed {SEED} '
             f'--input-type region --region-emb-source hgi '
             f'--override-hparams d_model=256 num_heads=8 '
             f'transition_path={TRANSITION_PATH} '
             f'--tag STL_{UPSTATE}_hgi_reg_gethard_{FOLDS}f{EPOCHS}ep')},

    # F36d — MTL B3 counterfactual (HGI substrate substituted into north-star B3)
    {'id': f'F36d_mtl_counterfactual_hgi_{UPSTATE}',
     'cmd': (f'python3 -u scripts/train.py --task mtl --state {STATE} --engine hgi '
             f'--task-set check2hgi_next_region --model mtlnet_crossattn '
             f'--mtl-loss static_weight --category-weight 0.75 '
             f'--reg-head next_getnext_hard --cat-head next_gru '
             f'--reg-head-param d_model=256 --reg-head-param num_heads=8 '
             f'--reg-head-param transition_path={TRANSITION_PATH} '
             f'--folds {FOLDS} --epochs {EPOCHS} --seed {SEED} --no-checkpoints')},
]

for exp in EXPERIMENTS:
    exp['log'] = LOGS_DIR / f"{STATE}_{exp['id']}.log"
    print(f"  {exp['id']}  →  {exp['log'].name}")

---
## ⑤ Sequential launcher

Runs each experiment as a detached subprocess (`nohup` + `start_new_session=True` per COLAB_GUIDE §5), polls until exit, then proceeds. Total wall-clock on T4 ≈ 3 h for FL (5× ~50min main runs + 2× ~5min probes).

**Resume-safe:** an experiment that already has its run dir on Drive is skipped (look for `results/<engine>/<state>/<run_dir>/summary/full_summary.json`).

In [ ]:
import subprocess, shlex, time, sys
from datetime import datetime

def already_done(exp):
    """Skip if a successful run JSON already exists on Drive (resume-safe)."""
    eid = exp['id']
    if 'F36a_probe' in eid:
        engine = 'check2hgi' if 'check2hgi' in eid else 'hgi'
        return (RESULTS_ROOT.parent / 'docs/studies/check2hgi/results/probe' /
                f'{STATE}_{engine}_last.json').exists()
    if 'F36b_cat_stl' in eid or 'F36d_mtl' in eid:
        engine = 'hgi' if ('hgi_' in eid and 'check2hgi' not in eid) else 'check2hgi'
        prefix = 'mtlnet_' if 'F36d' in eid else 'next_'
        rd = RESULTS_ROOT / engine / STATE
        return any(rd.glob(f'{prefix}*/summary/full_summary.json')) if rd.exists() else False
    if 'F36c_reg_stl' in eid:
        # p1_region_head_ablation writes a flat JSON at results/P1/region_head_<state>_*.json
        engine = 'hgi' if 'hgi' in eid and 'check2hgi' not in eid else 'check2hgi'
        tag = f'STL_{UPSTATE}_{engine}_reg_gethard_{FOLDS}f{EPOCHS}ep'
        return list((RESULTS_ROOT / 'P1').glob(f'*{tag}*.json')) != [] if (RESULTS_ROOT / 'P1').exists() else False
    return False

def run_one(exp):
    print(f"\n{'=' * 72}\n[{datetime.now().strftime('%H:%M:%S')}] START {exp['id']}\n{'=' * 72}")
    print(f'  cmd: {exp["cmd"]}')
    print(f'  log: {exp["log"]}')

    if already_done(exp):
        print(f'  SKIP — output already on Drive.')
        return 0

    subprocess.run(['pkill', '-9', '-f', 'scripts/train.py'], capture_output=True)
    subprocess.run(['pkill', '-9', '-f', 'p1_region_head_ablation.py'], capture_output=True)

    with exp['log'].open('w') as lf:
        proc = subprocess.Popen(
            ['nohup'] + shlex.split(exp['cmd']),
            stdout=lf, stderr=subprocess.STDOUT,
            cwd=str(REPO_DIR),
            start_new_session=True,
        )
    print(f'  PID {proc.pid} (detached)')

    # Poll until exit; print a heartbeat every 60 s with the last log line.
    while proc.poll() is None:
        time.sleep(60)
        text = exp['log'].read_text() if exp['log'].exists() else ''
        last = text.splitlines()[-1] if text.strip() else '(no output yet)'
        print(f'  [{datetime.now().strftime("%H:%M:%S")}] PID {proc.pid} alive — last: {last[:120]}')
        sys.stdout.flush()
    rc = proc.returncode
    print(f"\n[{datetime.now().strftime('%H:%M:%S')}] EXIT {exp['id']} rc={rc}")
    if rc != 0:
        print(f'  ⚠ non-zero exit — last 30 lines of log:')
        text = exp['log'].read_text() if exp['log'].exists() else ''
        for line in text.splitlines()[-30:]:
            print(f'    {line}')
    return rc

# Run the grid sequentially. Halt on first non-zero rc.
T0 = time.time()
for exp in EXPERIMENTS:
    rc = run_one(exp)
    if rc != 0:
        print(f'\n❌ Halting grid — {exp["id"]} failed.')
        break
else:
    print(f'\n✅ Phase-2 grid for {STATE} complete in {(time.time() - T0) / 60:.1f} min')

---
## ⑥ Monitor (re-run anytime to refresh)

Tails the latest log file and reports per-experiment status. Useful if you reconnect after a kernel disconnect — the launcher in §⑤ won't be running but the subprocess might still be.

In [ ]:
import subprocess
from datetime import datetime

def proc_alive(pattern):
    r = subprocess.run(['pgrep', '-f', pattern], capture_output=True, text=True)
    return bool(r.stdout.strip())

for exp in EXPERIMENTS:
    log = exp['log']
    if not log.exists():
        print(f"  ⚪ {exp['id']:<45} — not started")
        continue
    text = log.read_text()
    nlines = len(text.splitlines())
    last = text.splitlines()[-1] if text.strip() else '(empty)'
    pat = exp['cmd'].split()[1].split('/')[-1]  # script filename
    alive = proc_alive(f"{pat}.*{STATE}")
    state = '🟢 RUN' if alive else ('✅ DONE' if 'complete' in text.lower() or 'finished' in text.lower() else '⏸ STOP')
    print(f"  {state}  {exp['id']:<45} {nlines:>5} lines  ·  {last[:80]}")

---
## ⑦ Aggregate Phase-2 results

Reads per-fold JSONs from each experiment's run dir and emits the substrate-comparison summary table for this state. Compare side-by-side with `docs/studies/check2hgi/research/SUBSTRATE_COMPARISON_FINDINGS.md` (AL+AZ Phase-1 reference).

In [ ]:
import json, re
import numpy as np

def latest_summary(engine, prefix='next_'):
    rd = RESULTS_ROOT / engine / STATE
    runs = sorted(rd.glob(f'{prefix}*/summary/full_summary.json'),
                  key=lambda p: p.stat().st_mtime, reverse=True) if rd.exists() else []
    return runs[0] if runs else None

def per_fold_metric(run_summary_path, task, metric):
    """Extract per-fold metric from <run_dir>/folds/foldN_info.json."""
    folds_dir = run_summary_path.parent.parent / 'folds'
    out = []
    for i in range(FOLDS):
        fp = folds_dir / f'fold{i}_info.json'
        if not fp.exists():
            return None
        d = json.load(fp.open())
        m = d.get('diagnostic_best_epochs', {}).get(task, {}).get('metrics', {})
        if metric not in m:
            return None
        out.append(m[metric])
    return np.array(out)

def fmt(arr):
    if arr is None: return '   —    '
    return f'{arr.mean()*100:6.2f} ± {arr.std()*100:.2f}'

print(f'\n=== Phase-2 substrate-comparison summary — {STATE.upper()} ===\n')

# Linear probe (Leg I)
probe_dir = REPO_DIR / 'docs/studies/check2hgi/results/probe'
for engine in ('check2hgi', 'hgi'):
    p = probe_dir / f'{STATE}_{engine}_last.json'
    if p.exists():
        d = json.load(p.open())
        f1 = d.get('per_fold_macro_f1', d.get('macro_f1_per_fold', []))
        if f1:
            arr = np.array(f1)
            print(f'  Leg I  probe   {engine:<12} F1: {arr.mean()*100:6.2f} ± {arr.std()*100:.2f}')

# Cat STL matched-head (Leg II.1) — task=next, metric=f1
for engine in ('check2hgi', 'hgi'):
    sp = latest_summary(engine, prefix='next_')
    if sp:
        arr = per_fold_metric(sp, 'next', 'f1')
        print(f'  Leg II.1 cat   {engine:<12} F1: {fmt(arr)}')

# Reg STL matched-head (Leg II.2) — read p1_region_head_ablation JSON
for engine in ('check2hgi', 'hgi'):
    tag = f'STL_{UPSTATE}_{engine}_reg_gethard_{FOLDS}f{EPOCHS}ep'
    matches = list((RESULTS_ROOT / 'P1').glob(f'*{tag}*.json')) if (RESULTS_ROOT / 'P1').exists() else []
    if matches:
        d = json.load(matches[0].open())
        per_fold = d.get('heads', {}).get('next_getnext_hard', {}).get('per_fold', [])
        if per_fold:
            acc10 = np.array([f.get('acc10', 0) for f in per_fold])
            mrr   = np.array([f.get('mrr', 0)   for f in per_fold])
            print(f'  Leg II.2 reg   {engine:<12} Acc@10: {acc10.mean()*100:6.2f} ± {acc10.std()*100:.2f}  ·  MRR: {mrr.mean()*100:6.2f} ± {mrr.std()*100:.2f}')

# MTL B3 counterfactual (Leg III) — task=next_category + next_region
sp = latest_summary('hgi', prefix='mtlnet_')
if sp:
    cat_f1   = per_fold_metric(sp, 'next_category', 'f1')
    reg_acc  = per_fold_metric(sp, 'next_region',   'top10_acc_indist')
    print(f'  Leg III  MTL+HGI            cat F1: {fmt(cat_f1)}  ·  reg Acc@10_indist: {fmt(reg_acc)}')

print(f'\n→ Compare against AL+AZ in docs/studies/check2hgi/research/SUBSTRATE_COMPARISON_FINDINGS.md §1–§3.')

---
## ⑧ When this state is done

1. Verify per-experiment exit codes were 0 (§⑤ output) and the §⑦ summary populates all 4 legs.
2. Copy the per-fold JSONs onto the host repo at:
   - `docs/studies/check2hgi/results/phase1_perfold/<UPSTATE>_<engine>_cat_gru_5f50ep.json` (cat STL — extract from `<run_dir>/folds/foldN_info.json::diagnostic_best_epochs.next.metrics`)
   - `docs/studies/check2hgi/results/P1/region_head_<state>_*.json` (reg STL — already in this format)
   - `docs/studies/check2hgi/results/probe/<state>_<engine>_last.json` (probe outputs already there if scripts/probe wrote to repo dir)
   - `docs/studies/check2hgi/results/phase1_perfold/<UPSTATE>_hgi_mtl_{cat,reg}.json` (MTL counterfactual)
3. Run `scripts/analysis/substrate_paired_test.py` for cat F1 + reg Acc@10/MRR + TOST δ=2pp (per `PHASE2_TRACKER.md §5` acceptance).
4. Update `PHASE2_TRACKER.md` status board: 🔴 → ✅ for this state's row.
5. Set `STATE` to the next state in §④ and re-run the notebook from §④ onward.